# Entrenamiento PredRNN v2 (TensorFlow) — Dataset KTH

Este notebook recorre paso a paso el ciclo de entrenamiento completo:
loader KTH → scheduled sampling → paso de train → evaluacion → visualizacion.

Sigue la misma logica que `predrnn_tf/train.py` pero de forma interactiva y explicada.

## 0. Paths e imports

In [1]:
import sys
import os
import math
import datetime
import numpy as np
import matplotlib.pyplot as plt

# Raiz del proyecto (predrnn-pytorch/)
PROJECT_ROOT = os.path.abspath('.')

# Para importar kth_action y preprocess del lado PyTorch
sys.path.insert(0, PROJECT_ROOT)

# Para importar el modelo TF y su model_factory
TF_ROOT = os.path.join(PROJECT_ROOT, 'predrnn_tf')
sys.path.insert(0, TF_ROOT)

# Path al dataset KTH
KTH_DATA_PATH = '/Users/emilioespinosa/Documents/DMREF_Project/Codigo/predrnn-pytorch/KTH'  # <-- CAMBIA ESTO

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TF_ROOT:     ', TF_ROOT)
print('KTH_DATA_PATH:', KTH_DATA_PATH)

PROJECT_ROOT: /Users/emilioespinosa/Documents/DMREF_Project/Codigo/predrnn-pytorch
TF_ROOT:      /Users/emilioespinosa/Documents/DMREF_Project/Codigo/predrnn-pytorch/predrnn_tf
KTH_DATA_PATH: /Users/emilioespinosa/Documents/DMREF_Project/Codigo/predrnn-pytorch/KTH


## 1. Configuracion de hiperparametros

Equivalente a los argumentos de `train.py`.  
Se usa `SimpleNamespace` para acceder con `configs.batch_size`, igual que `args.batch_size` en el script.

In [2]:
from types import SimpleNamespace

configs = SimpleNamespace(
    # --- Datos ---
    img_width    = 128,   # resolucion original del frame (cuadrado)
    img_channel  = 1,     # escala de grises
    input_length = 10,    # frames que ve el modelo como contexto
    total_length = 20,    # frames totales por secuencia (10 entrada + 10 prediccion)

    # --- Modelo ---
    model_name   = 'predrnn_v2',
    num_hidden   = '64,64,64,64',   # neuronas por capa ST-LSTM
    filter_size  = 5,               # tamano del kernel convolucional
    stride       = 1,
    patch_size   = 4,               # factor de reduccion espacial
    layer_norm   = 1,
    decouple_beta = 0.1,            # peso de la decouple loss

    # --- Scheduled Sampling ---
    scheduled_sampling    = 1,
    sampling_stop_iter    = 50000,
    sampling_start_value  = 1.0,    # empieza usando 100% frames reales
    sampling_changing_rate = 0.00002,
    reverse_scheduled_sampling = 0,
    # (parametros de reverse sampling, no usados si reverse=0)
    r_sampling_step_1 = 25000,
    r_sampling_step_2 = 50000,
    r_exp_alpha       = 5000,

    # --- Optimizacion ---
    lr               = 0.001,
    batch_size       = 4,
    max_iterations   = 500,         # pocas iteraciones para explorar
    reverse_input    = 1,           # entrenar tambien con secuencia invertida

    # --- Logging y guardado ---
    display_interval  = 50,
    test_interval     = 200,
    snapshot_interval = 200,
    num_save_samples  = 3,
    save_dir          = 'checkpoints/kth_predrnn_v2_tf',
    gen_frm_dir       = 'results/kth_predrnn_v2_tf',
)

# Dimensiones derivadas (utiles para referencia)
H_patch = configs.img_width // configs.patch_size
C_patch = configs.patch_size ** 2 * configs.img_channel
n_pred  = configs.total_length - configs.input_length

print(f'Frame original:    [{configs.img_width} x {configs.img_width} x {configs.img_channel}]')
print(f'Frame en patches:  [{H_patch} x {H_patch} x {C_patch}]')
print(f'Frames de entrada: {configs.input_length}')
print(f'Frames predichos:  {n_pred}')

Frame original:    [128 x 128 x 1]
Frame en patches:  [32 x 32 x 16]
Frames de entrada: 10
Frames predichos:  10


## 2. Cargar datos KTH

Reutilizamos el loader que ya exploramos en `explore_kth_loader.ipynb`.

In [3]:
from core.data_provider import kth_action

input_param = {
    'paths':           [KTH_DATA_PATH],
    'image_width':     configs.img_width,
    'minibatch_size':  configs.batch_size,
    'seq_length':      configs.total_length,
    'input_data_type': 'float32',
    'name':            'kth_train',
}

print('Instanciando DataProcess...')
data_process = kth_action.DataProcess(input_param)
print('Categorias:', data_process.categories)

print('\nCargando datos de TRAIN...')
train_handle = data_process.get_train_input_handle()
train_handle.begin(do_shuffle=True)

print('\nCargando datos de TEST...')
test_handle = data_process.get_test_input_handle()
test_handle.begin(do_shuffle=False)

print(f'\nTrain: {train_handle.total()} secuencias')
print(f'Test:  {test_handle.total()} secuencias')

Instanciando DataProcess...
Categorias: ['boxing', 'running']

Cargando datos de TRAIN...
begin load data/Users/emilioespinosa/Documents/DMREF_Project/Codigo/predrnn-pytorch/KTH
Preparing to load 54545 video frames.
there are 54545 pictures
there are 9309 sequences

Cargando datos de TEST...
begin load data/Users/emilioespinosa/Documents/DMREF_Project/Codigo/predrnn-pytorch/KTH
Preparing to load 29147 video frames.
there are 29147 pictures
there are 4935 sequences

Train: 9309 secuencias
Test:  4935 secuencias


## 3. Construir el modelo TensorFlow

`Model` (de `model_factory.py`) encapsula:
- La red `RNN` (PredRNN v2)
- El optimizador Adam
- Los metodos `train()`, `test()`, `save()`, `load()`

In [4]:
import tensorflow as tf
from core.models.model_factory import Model

print('Version TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

print('\nConstruyendo modelo...')
model = Model(configs)

num_hidden = [int(x) for x in configs.num_hidden.split(',')]
print(f'Arquitectura:')
print(f'  Capas ST-LSTM:  {len(num_hidden)}')
print(f'  Neuronas/capa:  {num_hidden}')
print(f'  Kernel:         {configs.filter_size}x{configs.filter_size}')
print(f'  Decouple beta:  {configs.decouple_beta}')

Version TensorFlow: 2.21.0
GPU disponible: []

Construyendo modelo...
Arquitectura:
  Capas ST-LSTM:  4
  Neuronas/capa:  [64, 64, 64, 64]
  Kernel:         5x5
  Decouple beta:  0.1


## 4. Scheduled Sampling — que es y como funciona

**Problema:** En entrenamiento, el modelo siempre ve frames reales como entrada.  
En test, solo tiene sus propias predicciones. Esto causa un *distribution shift*.

**Scheduled Sampling:** Gradualmente reemplaza frames reales por predicciones del modelo durante el entrenamiento.

La mascara `real_input_flag` de forma `[B, T-input_length-1, H_patch, W_patch, C_patch]` controla esto:
- `1` en posicion `t` → usar frame **real** en ese paso
- `0` en posicion `t` → usar **prediccion** del modelo

Al inicio (`eta=1.0`) → todo unos → 100% frames reales  
Al final (`eta=0.0`) → todo ceros → 100% predicciones propias

In [ ]:
def schedule_sampling(eta, itr, configs):
    """
    Scheduled sampling estandar (forward).
    
    Retorna:
        eta_new          : float, nuevo valor de eta (decrece con cada iteracion)
        real_input_flag  : np.array [B, T-input_length-1, H_p, W_p, C_p]
                           1 = usar frame real, 0 = usar prediccion del modelo
    """
    patch_shape = (
        configs.img_width // configs.patch_size,
        configs.img_width // configs.patch_size,
        configs.patch_size ** 2 * configs.img_channel,
    )
    n_future = configs.total_length - configs.input_length - 1
    zeros_mask = np.zeros((configs.batch_size, n_future, *patch_shape), dtype=np.float32)

    if not configs.scheduled_sampling:
        return 0.0, zeros_mask

    # Decrecer eta hasta llegar a 0
    if itr < configs.sampling_stop_iter:
        eta -= configs.sampling_changing_rate
    else:
        eta = 0.0

    # Para cada secuencia del batch y cada frame futuro:
    # tirar una moneda con probabilidad eta de ser real
    random_flip = np.random.random_sample((configs.batch_size, n_future))
    true_token  = (random_flip < eta)  # True = usar frame real

    ones  = np.ones(patch_shape)
    zeros = np.zeros(patch_shape)

    real_input_flag = []
    for b in range(configs.batch_size):
        for t in range(n_future):
            real_input_flag.append(ones if true_token[b, t] else zeros)

    real_input_flag = np.reshape(
        np.array(real_input_flag),
        (configs.batch_size, n_future, *patch_shape)
    ).astype(np.float32)

    return eta, real_input_flag


# --- Visualizar como evoluciona la mascara con eta ---
etas = [1.0, 0.7, 0.4, 0.1, 0.0]
fig, axes = plt.subplots(1, len(etas), figsize=(14, 2))

for ax, eta_val in zip(axes, etas):
    np.random.seed(42)
    n_future = configs.total_length - configs.input_length - 1
    flip = np.random.random_sample((configs.batch_size, n_future))
    mask = (flip < eta_val).astype(float)
    ax.imshow(mask, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'eta={eta_val}', fontsize=9)
    ax.set_xlabel('frame futuro t')
    ax.set_ylabel('batch b')

plt.suptitle('real_input_flag (azul=real, blanco=prediccion)', fontsize=10)
plt.tight_layout()
plt.show()

print('n_future (frames con mascara):', n_future)
print('Forma de real_input_flag: [B=%d, T_fut=%d, H_p=%d, W_p=%d, C_p=%d]' %
      (configs.batch_size, n_future,
       configs.img_width//configs.patch_size,
       configs.img_width//configs.patch_size,
       configs.patch_size**2 * configs.img_channel))

## 5. Un paso de entrenamiento manual

Reproducimos exactamente lo que hace `train_wrapper` en `train.py` para una sola iteracion:  
`get_batch` → `reshape_patch` → `schedule_sampling` → `model.train()`

In [ ]:
from core.utils import preprocess

# --- Paso 1: obtener batch del loader ---
raw_batch = train_handle.get_batch()                              # [B, T, H, W, C]
print(f'raw_batch shape: {raw_batch.shape}  <- [B, T, H, W, C]')

# --- Paso 2: transformar al espacio de patches ---
ims = preprocess.reshape_patch(raw_batch, configs.patch_size)    # [B, T, H_p, W_p, C_p]
print(f'ims (patches):   {ims.shape}  <- [B, T, H_p, W_p, C_p]')

# --- Paso 3: crear la mascara de scheduled sampling ---
eta = configs.sampling_start_value
eta, real_input_flag = schedule_sampling(eta, itr=1, configs=configs)
print(f'real_input_flag: {real_input_flag.shape}')
print(f'eta inicial -> nuevo: {configs.sampling_start_value:.4f} -> {eta:.4f}')
print(f'% frames reales en la mascara: {real_input_flag.mean()*100:.1f}%')

# --- Paso 4: un paso de entrenamiento ---
# model.train() internamente:
#   1. Crea tf.Tensor desde numpy
#   2. tf.GradientTape -> forward pass (call) -> loss
#   3. tape.gradient -> optimizer.apply_gradients
#   4. Retorna loss como float
cost = model.train(ims, real_input_flag)
print(f'\nLoss primer paso: {cost:.6f}')

# --- Paso 5 (opcional): entrenar tambien con secuencia invertida ---
if configs.reverse_input:
    ims_rev = np.flip(ims, axis=1).copy()   # invertir dimension temporal
    cost_rev = model.train(ims_rev, real_input_flag)
    cost = (cost + cost_rev) / 2.0
    print(f'Loss invertido:   {cost_rev:.6f}')
    print(f'Loss promedio:    {cost:.6f}')

## 6. Inicializar variables del modelo con un forward pass

TensorFlow crea las variables la primera vez que se ejecuta `call()`.  
Esto ya ocurrio en el paso anterior. Aqui verificamos cuantos parametros tiene el modelo.

In [ ]:
total_params = sum(np.prod(v.shape) for v in model.network.trainable_variables)
print(f'Total parametros entrenables: {total_params:,}')
print(f'  ~{total_params/1e6:.2f}M parametros')

print('\nVariables por capa:')
for v in model.network.trainable_variables:
    print(f'  {v.name:50s}  shape={str(v.shape):20s}  params={np.prod(v.shape):,}')

## 7. Bucle de entrenamiento completo

Replica exactamente `train_wrapper()` de `train.py`:

```
for itr in range(1, max_iterations+1):
    if no hay batch -> begin(shuffle)
    ims = get_batch -> reshape_patch
    real_input_flag = schedule_sampling(eta, itr)
    trainer.train(model, ims, real_input_flag, configs, itr)
    if itr % snapshot_interval -> model.save()
    if itr % test_interval     -> trainer.test()
    train_handle.next()
```

In [ ]:
import core.trainer as trainer_tf

os.makedirs(configs.save_dir,    exist_ok=True)
os.makedirs(configs.gen_frm_dir, exist_ok=True)

eta = configs.sampling_start_value
loss_history = []

print(f'Iniciando entrenamiento: {configs.max_iterations} iteraciones')
print(f'  batch_size={configs.batch_size}, seq_length={configs.total_length}')
print(f'  scheduled_sampling: eta empieza en {eta}, decae {configs.sampling_changing_rate}/iter')
print()

for itr in range(1, configs.max_iterations + 1):

    # --- Reiniciar el handle si se agotaron los batches (nueva epoca) ---
    if train_handle.no_batch_left():
        train_handle.begin(do_shuffle=True)

    # --- Obtener batch y transformar a espacio de patches ---
    ims = train_handle.get_batch()                             # [B, T, H, W, C]
    ims = preprocess.reshape_patch(ims, configs.patch_size)   # [B, T, H_p, W_p, C_p]

    # --- Calcular mascara de scheduled sampling ---
    if configs.reverse_scheduled_sampling == 1:
        # Reverse: empieza con predicciones, termina con reales
        real_input_flag = reserve_schedule_sampling_exp(itr, configs)
    else:
        # Forward: empieza con reales, mezcla predicciones gradualmente
        eta, real_input_flag = schedule_sampling(eta, itr, configs)

    # --- Un paso de entrenamiento ---
    cost = model.train(ims, real_input_flag)
    if configs.reverse_input:
        ims_rev = np.flip(ims, axis=1).copy()
        cost   += model.train(ims_rev, real_input_flag)
        cost   /= 2.0

    loss_history.append(cost)

    # --- Log periodico ---
    if itr % configs.display_interval == 0:
        ts = datetime.datetime.now().strftime('%H:%M:%S')
        print(f'[{ts}] itr {itr:5d}/{configs.max_iterations}  '
              f'loss={cost:.6f}  eta={eta:.4f}')

    # --- Guardar checkpoint ---
    if itr % configs.snapshot_interval == 0:
        model.save(itr)

    # --- Evaluacion en test ---
    if itr % configs.test_interval == 0:
        trainer_tf.test(model, test_handle, configs, itr)

    # --- Avanzar al siguiente batch ---
    train_handle.next()

print('\nEntrenamiento completado.')

## 8. Curva de loss durante el entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Curva completa
axes[0].plot(loss_history, linewidth=0.8, color='steelblue')
axes[0].set_xlabel('Iteracion')
axes[0].set_ylabel('Loss (MSE + decouple)')
axes[0].set_title('Loss de entrenamiento')
axes[0].grid(True, alpha=0.3)

# Suavizado (media movil)
window = max(1, len(loss_history) // 20)
smooth = np.convolve(loss_history, np.ones(window)/window, mode='valid')
axes[1].plot(smooth, linewidth=1.5, color='darkorange')
axes[1].set_xlabel('Iteracion')
axes[1].set_ylabel('Loss (suavizado)')
axes[1].set_title(f'Loss suavizado (ventana={window})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Loss inicial:  {loss_history[0]:.6f}')
print(f'Loss final:    {loss_history[-1]:.6f}')
print(f'Loss minimo:   {min(loss_history):.6f}')

## 9. Evaluacion manual sobre test

Replica `trainer.test()` pero de forma explicita para ver cada paso.

En test la mascara es **todo ceros**: el modelo no ve ningun frame futuro real,
solo usa sus propias predicciones encadenadas.

In [ ]:
from skimage.metrics import structural_similarity as compare_ssim
from core.utils import metrics

# Mascara de test = todo ceros (modelo predice libre, sin ver frames reales futuros)
mask_input = configs.input_length
real_input_flag_test = np.zeros((
    configs.batch_size,
    configs.total_length - mask_input - 1,
    configs.img_width  // configs.patch_size,
    configs.img_width  // configs.patch_size,
    configs.patch_size ** 2 * configs.img_channel
), dtype=np.float32)

print(f'Mascara de test: shape={real_input_flag_test.shape}')
print(f'Suma total (debe ser 0): {real_input_flag_test.sum()}')
print()

# Reiniciar test handle
test_handle.begin(do_shuffle=False)

n_pred    = configs.total_length - configs.input_length
avg_mse   = 0.0
batch_id  = 0
img_mse   = [0.0] * n_pred
ssim_vals = [0.0] * n_pred
psnr_vals = [0.0] * n_pred

# Guardar el primer batch para visualizar
saved_batch = None
saved_gen   = None

while not test_handle.no_batch_left():
    batch_id += 1

    # --- Obtener batch de test ---
    test_ims_raw = test_handle.get_batch()                                    # [B, T, H, W, C]
    test_dat     = preprocess.reshape_patch(test_ims_raw, configs.patch_size) # [B, T, H_p, W_p, C_p]
    test_ims     = test_ims_raw[:, :, :, :, :configs.img_channel]             # asegurar canales correctos

    # --- Inferencia ---
    img_gen = model.test(test_dat, real_input_flag_test)                      # [B, T-1, H_p, W_p, C_p]
    img_gen = preprocess.reshape_patch_back(img_gen, configs.patch_size)      # [B, T-1, H, W, C]
    img_out = img_gen[:, -n_pred:]                                            # solo los frames predichos

    # Guardar primer batch para visualizar
    if batch_id == 1:
        saved_batch = test_ims.copy()
        saved_gen   = img_out.copy()

    # --- Metricas por frame predicho ---
    for i in range(n_pred):
        x  = test_ims[:, i + configs.input_length, :, :, :]   # ground truth
        gx = np.clip(img_out[:, i, :, :, :], 0.0, 1.0)        # prediccion

        mse = np.square(x - gx).sum()
        img_mse[i] += mse
        avg_mse    += mse

        real_frm = np.uint8(x  * 255)
        pred_frm = np.uint8(gx * 255)

        psnr_vals[i] += metrics.batch_psnr(pred_frm, real_frm)

        for b in range(configs.batch_size):
            score, _ = compare_ssim(
                pred_frm[b], real_frm[b],
                full=True, multichannel=True
            )
            ssim_vals[i] += score

    test_handle.next()

# --- Resumen ---
norm = batch_id * configs.batch_size
avg_mse /= norm
ssim_arr = np.array(ssim_vals) / norm
psnr_arr = np.array(psnr_vals) / batch_id

print(f'Batches de test evaluados: {batch_id}')
print(f'MSE promedio por secuencia: {avg_mse:.6f}')
print(f'SSIM promedio:              {np.mean(ssim_arr):.4f}')
print(f'PSNR promedio:              {np.mean(psnr_arr):.4f} dB')

## 10. Metricas por frame predicho

Cuanto peor predice el modelo a medida que avanza en el tiempo (frames mas lejanos = mas dificiles).

In [ ]:
frame_ids = list(range(1, n_pred + 1))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

mse_per_frame = np.array(img_mse) / norm

axes[0].plot(frame_ids, mse_per_frame, marker='o', color='steelblue')
axes[0].set_xlabel('Frame predicho')
axes[0].set_ylabel('MSE')
axes[0].set_title('MSE por frame')
axes[0].grid(True, alpha=0.3)

axes[1].plot(frame_ids, ssim_arr, marker='o', color='darkorange')
axes[1].set_xlabel('Frame predicho')
axes[1].set_ylabel('SSIM')
axes[1].set_title('SSIM por frame')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

axes[2].plot(frame_ids, psnr_arr, marker='o', color='seagreen')
axes[2].set_xlabel('Frame predicho')
axes[2].set_ylabel('PSNR (dB)')
axes[2].set_title('PSNR por frame')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Metricas de evaluacion — KTH ({n_pred} frames predichos)', fontsize=11)
plt.tight_layout()
plt.show()

## 11. Visualizar predicciones vs ground truth

Fila superior: frames reales (ground truth)  
Fila inferior: frames predichos por el modelo  
Linea vertical separa frames de contexto (entrada) de frames predichos

In [ ]:
# Mostrar el primer video del primer batch de test
b_idx = 0

gt_frames   = saved_batch[b_idx]   # [T, H, W, 1]  ground truth completo
pred_frames = saved_gen[b_idx]     # [n_pred, H, W, 1]  predicciones

T = configs.total_length
cols = T

fig, axes = plt.subplots(2, cols, figsize=(cols * 1.3, 3))

for t in range(T):
    # Fila 0: ground truth
    axes[0, t].imshow(gt_frames[t, :, :, 0], cmap='gray', vmin=0, vmax=1)
    axes[0, t].axis('off')
    axes[0, t].set_title(f't={t}', fontsize=6)

    # Fila 1: prediccion (solo para frames futuros)
    if t >= configs.input_length:
        pred_t = t - configs.input_length
        frame  = np.clip(pred_frames[pred_t, :, :, 0], 0, 1)
        axes[1, t].imshow(frame, cmap='gray', vmin=0, vmax=1)
    else:
        axes[1, t].imshow(np.zeros((configs.img_width, configs.img_width)),
                          cmap='gray', vmin=0, vmax=1)
        axes[1, t].text(0.5, 0.5, 'entrada', ha='center', va='center',
                        transform=axes[1, t].transAxes, fontsize=6, color='white')
    axes[1, t].axis('off')

# Linea divisoria entre entrada y prediccion
for row in range(2):
    axes[row, configs.input_length - 1].set_title(
        f't={configs.input_length-1}\n<-entrada | pred->', fontsize=5)

axes[0, 0].set_ylabel('Ground\ntruth', fontsize=8)
axes[1, 0].set_ylabel('Prediccion', fontsize=8)

plt.suptitle(f'Ground truth vs Prediccion — secuencia de test (video {b_idx})', fontsize=10)
plt.tight_layout()
plt.show()

## 12. Error por pixel (mapa de diferencias)

Visualiza donde se equivoca el modelo: zonas brillantes = error alto.

In [ ]:
# Mostrar los primeros 5 frames predichos y su error absoluto
n_show = min(5, n_pred)

fig, axes = plt.subplots(3, n_show, figsize=(n_show * 2.5, 6))

for i in range(n_show):
    t_gt = configs.input_length + i

    gt   = gt_frames[t_gt, :, :, 0]
    pred = np.clip(pred_frames[i, :, :, 0], 0, 1)
    err  = np.abs(gt - pred)

    axes[0, i].imshow(gt,   cmap='gray',  vmin=0, vmax=1)
    axes[0, i].set_title(f'GT t={t_gt}', fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(pred, cmap='gray',  vmin=0, vmax=1)
    axes[1, i].set_title(f'Pred t={t_gt}', fontsize=8)
    axes[1, i].axis('off')

    im = axes[2, i].imshow(err,  cmap='hot',   vmin=0, vmax=0.3)
    axes[2, i].set_title(f'|Error| max={err.max():.3f}', fontsize=8)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Ground truth', fontsize=8)
axes[1, 0].set_ylabel('Prediccion',   fontsize=8)
axes[2, 0].set_ylabel('Error abs.',   fontsize=8)

plt.suptitle('Mapas de error absoluto por frame predicho', fontsize=10)
plt.tight_layout()
plt.show()

## 13. Guardar y cargar checkpoint

Muestra como guardar el modelo y recargarlo para continuar entrenamiento o inferencia.

In [ ]:
# --- Guardar ---
model.save('final')

# --- Cargar en un modelo nuevo ---
model2 = Model(configs)

# Warm-up: necesario para que TF inicialice las variables antes de cargar pesos
ph = configs.img_width // configs.patch_size
cp = configs.patch_size ** 2 * configs.img_channel
dummy_frames = np.zeros((configs.batch_size, configs.total_length, ph, ph, cp), dtype=np.float32)
dummy_mask   = np.zeros((configs.batch_size, configs.total_length - configs.input_length - 1, ph, ph, cp), dtype=np.float32)
model2.test(dummy_frames, dummy_mask)   # forward pass dummy

# Cargar pesos
checkpoint_path = os.path.join(configs.save_dir, 'model-final')
model2.load(checkpoint_path)

# Verificar que produce los mismos resultados
test_handle.begin(do_shuffle=False)
sample = test_handle.get_batch()
sample_patches = preprocess.reshape_patch(sample, configs.patch_size)

out1 = model.test(sample_patches,  real_input_flag_test)
out2 = model2.test(sample_patches, real_input_flag_test)

max_diff = np.abs(out1 - out2).max()
print(f'Diferencia maxima entre modelo original y recargado: {max_diff:.2e}  <- debe ser ~0')